# CropFusion — System check (R2.1)

Kaggle notebook that validates the entire Training Platform infrastructure:
runtime, Python, CUDA, GPU, dependencies, folder structure, permissions, disk
space and dataset providers. Generates the environment / GPU / dependency /
storage / workspace / configuration / validation reports.

- Dataset (attach this notebook to it): `shathanandabhatn/crop-yield-forecasting-karnataka-dakshina-kannada`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/Brijesh2005/CropPrep.git'
REPO_ROOT = Path('/kaggle/working/CropPrep')

if not (REPO_ROOT / '.git').exists():
    print(f'cloning CropPrep -> {REPO_ROOT}')
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')
print(f'cwd: {os.getcwd()}')


## 1.1 P100 GPU fix

Kaggle's base PyTorch (cu128) dropped Pascal `sm_60` kernels, so the P100
cannot execute any kernel (`CUDA error: no kernel image is available`).
Reinstall torch from the cu126 index, which still ships `sm_60` cubins
(verified fix, kaggle/docker-python#1546).


In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'],
    check=False,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126'],
    check=True,
)
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      '| arch', torch.cuda.get_arch_list())


## 1. Bootstrap

Same environment + data-source setup as the other notebooks.

In [ ]:
!python training/kaggle/scripts/bootstrap.py --skip-install

## 2. Validate + report

Runs the Training Validator and writes every report under
`training/kaggle/outputs/reports/`. Exit code 0 means the infrastructure is ready.

In [ ]:
!python training/kaggle/scripts/system_check.py; echo "system_check_exit=$?"


## 3. Inspect the validation report

Shows every finding grouped by severity.

In [ ]:
import json
from pathlib import Path

report_path = REPO_ROOT / 'training' / 'kaggle' / 'outputs' / 'reports' / 'validation.json'
if report_path.exists():
    data = json.loads(report_path.read_text(encoding='utf-8'))
    print('passed:', data['passed'])
    for issue in data['issues']:
        print(f"  [{issue['severity']:8s}] {issue['code']}: {issue['message']}")
else:
    print('validation report not found at', report_path)